# Notebook 008: Sorting, Ordering Semantics & Stratified Sampling

**Module:** Module 01: Core PySpark Foundations & DataFrame API  
**Difficulty:** `Beginner`  
**Databricks Masterclass Reference:** Video 13: Databricks Clusters & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
A financial auditing team must sample transaction logs for regulatory compliance. The sample must be stratified across risk tiers (Low, Medium, High) to reflect true operational risk distribution without skewing toward high-volume low-risk transactions.

## 2. Technical Requirement & Problem Statement
Implement ordering with explicit null placement (`asc_nulls_first`, `desc_nulls_last`). Compare `sample()` vs `sampleBy()` for stratified sampling across categorical groups, and evaluate execution performance.

## 3. Spark & Databricks Architecture Notes
`orderBy()` and `sort()` trigger a global RangePartitioner shuffle across all executor partitions, which can be expensive on multi-terabyte datasets. `sortWithinPartitions()` sorts locally inside each partition without shuffling.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_008_Sorting,_Ordering_Semantics_&_Stratified_Sampling")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Creating Auditing Dataset
Generate audit logs with risk tiers and financial values.

In [ ]:
import random

# Seed for reproducible mock data
random.seed(42)
risk_tiers = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

records = [
    (
        f"AUD-{i:04d}",
        random.choice(risk_tiers),
        round(random.uniform(10.0, 50000.0), 2) if i % 10 != 0 else None,
        f"ORG-{random.randint(1, 5)}"
    )
    for i in range(1, 101)
]

df_audit_logs = spark.createDataFrame(records, ["audit_id", "risk_tier", "amount", "org_id"])
df_audit_logs.show(5)

## 6. Multi-Column Sorting with Explicit Null Placement
Sort by risk tier and amount, ensuring null amounts are placed at the end.

In [ ]:
df_sorted = df_audit_logs.orderBy(
    F.col("risk_tier").asc(),
    F.col("amount").desc_nulls_last()
)

print("Top 10 Sorted Records:")
df_sorted.show(10)

## 7. Stratified Sampling via sampleBy
Extract representative samples per risk tier using exact fractions.

In [ ]:
# Define stratification fractions: sample 10% of LOW, 30% of MEDIUM, 80% of HIGH, 100% of CRITICAL
fractions = {
    "LOW": 0.10,
    "MEDIUM": 0.30,
    "HIGH": 0.80,
    "CRITICAL": 1.00
}

df_stratified = df_audit_logs.sampleBy("risk_tier", fractions=fractions, seed=42)

print("Stratified Sample Distribution:")
df_stratified.groupBy("risk_tier").count().show()

## 9. Key Architectural Takeaways
- **`orderBy` performs a global sort via RangePartitioner shuffle, while `sortWithinPartitions` performs local memory sorts without shuffle.**
- **`sampleBy()` performs stratified sampling without requiring manual filtering and union steps.**
- **Always specify `seed` in sampling calls when building deterministic test suites or reproducible ML splits.**

## 10. Senior Data Engineering Interview Questions
1. What is the difference between `sort()` and `sortWithinPartitions()` in PySpark?
2. How does Spark's RangePartitioner sample keys to determine partition boundaries during a global sort?
3. Why can `df.sample(withReplacement=False, fraction=0.1)` produce slightly more or less than 10% of total rows?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
